In [ ]:
import os
from datetime import UTC, datetime, timedelta

import cabaret
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits

In [ ]:
vignette = fits.getdata(
    "../data/images/20250410/camera_hpp_G_FLAT_1.781_20250410_181814.820.fits"
).astype(np.uint16)

vignette = vignette - 510  # subtract the bias level
vignette = vignette / np.median(vignette)
print(min(vignette.flatten()), max(vignette.flatten()))
plt.imshow(vignette, cmap="gray")
plt.colorbar()

In [ ]:
## Observatory components

site = cabaret.Site(
    sky_background=500,  # e-/m^2/arcsec^2/s
    seeing=3.5,  # arcseconds
    elevation=500,  # meters
)

telescope = cabaret.Telescope(
    focal_length=3.454,  # meters
    diameter=0.5,  # meters
    collecting_area=0.233,  # calculated from diameter
)

camera = cabaret.Camera(
    name="Simulated Moravian C5A-100M",
    width=11664 // 4,  # pixels
    height=8750 // 4,  # pixels
    bin_x=1,  # pixels
    bin_y=1,  # pixels
    pitch=3.76 * 4,  # microns
    plate_scale=None,  # arcseconds per pixel (calculated from pitch and telescope)
    max_adu=65535,  # ADU
    well_depth=65535,  # electrons
    bias=500,  # ADU
    gain=1,  # electrons per ADU
    read_noise=2.1,  # electrons
    dark_current=0.0008,  # electrons per second
    average_quantum_efficiency=1,  # fraction
    rotation=0,  # degrees
    pixel_defects={
        "light_distribution": {
            "type": "light_distribution",
            "distribution_map": vignette,
        },
    },
)

## Define coordinates for the center of the image

CLUSTERS = {
    "M45": dict(
        ra=56.75,
        dec=24.12,
        radius=1.5,
        dist=135.0,
        age="~125 Myr",
        A_G=0.12,
        E_BPRP=0.06,
        exp_time=1,  # seconds
    ),
    "M44": dict(
        ra=130.10,
        dec=19.67,
        radius=1.2,
        dist=185.0,
        age="~650 Myr",
        A_G=0.07,
        E_BPRP=0.04,
        exp_time=1,  # seconds
    ),
    "M67": dict(
        ra=132.846,
        dec=11.814,
        radius=0.6,
        dist=850.0,
        age="~4 Gyr",
        A_G=0.11,
        E_BPRP=0.05,
        exp_time=1,  # seconds
    ),
    "NGC188": dict(
        ra=11.83,
        dec=85.244,
        radius=0.5,
        dist=1700.0,
        age="~7 Gyr",
        A_G=0.23,
        E_BPRP=0.12,
        exp_time=1,  # seconds
    ),
}


In [ ]:
observatory = cabaret.Observatory(
    name="Simulated ETH Observatory",
    site=site,
    telescope=telescope,
    camera=camera,
)

camera.set_plate_scale_from_focal_length(telescope.focal_length)

In [ ]:
from astroquery.gaia import Gaia

Gaia.ROW_LIMIT = -1
DIST_FRAC = 0.10  # +/-10% distance band


def query(p):
    """Cone search of Gaia DR3."""
    adql = f"""
    SELECT ra, dec, parallax, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag
    FROM gaiadr3.gaia_source
    WHERE 1 = CONTAINS(POINT('ICRS', ra, dec),
                       CIRCLE('ICRS', {p["ra"]}, {p["dec"]}, {p["radius"]}))
      AND parallax IS NOT NULL
      AND phot_bp_mean_mag IS NOT NULL
      AND phot_rp_mean_mag IS NOT NULL
      AND ruwe < 1.4
    """
    return Gaia.launch_job_async(adql).get_results()


def members(t, p, frac=DIST_FRAC):
    """Keep stars whose parallax distance is within +/-frac of the expected
    distance. Done in parallax space (monotonic) so bad/negative parallaxes
    fall out automatically.
        d in [(1-f) d0, (1+f) d0]  <=>  plx in [1000/((1+f)d0), 1000/((1-f)d0)]
    """
    plx = np.asarray(t["parallax"], float)
    plx_lo = 1000.0 / ((1 + frac) * p["dist"])  # far edge -> smaller parallax
    plx_hi = 1000.0 / ((1 - frac) * p["dist"])  # near edge -> larger parallax
    return t[(plx >= plx_lo) & (plx <= plx_hi)], (plx_lo, plx_hi)

In [ ]:
from cabaret.queries import GaiaQuery

cluster_name = "M67"

ra = CLUSTERS[cluster_name]["ra"]
dec = CLUSTERS[cluster_name]["dec"]
exp_time = CLUSTERS[cluster_name]["exp_time"]
n_imgs = 10

filters = [cabaret.Filters.G, cabaret.Filters.RP, cabaret.Filters.BP]

# Light frames
os.makedirs(f"../data/{cluster_name}", exist_ok=True)
original_table = query(CLUSTERS[cluster_name])
for i in range(len(filters)):
    center = SkyCoord(ra=ra, dec=dec, unit="deg")
    dateobs = datetime.now(UTC)

    table = members(original_table, CLUSTERS[cluster_name])[0]

    fluxes = GaiaQuery._mag_to_photons(
        np.ma.filled(table[filters[i].value].value, np.nan),  # type: ignore
        filters[i],
    )
    table.remove_rows(np.isnan(fluxes))
    fluxes = fluxes[~np.isnan(fluxes)]
    sources = cabaret.Sources.from_arrays(
        ra=table["ra"].value, dec=table["dec"].value, fluxes=fluxes
    )

    # Generate the FITS image for this filter
    for j in range(n_imgs):
        dateobs = datetime.now(UTC) + timedelta(seconds=j * (exp_time + 1))

        ## Light frame
        hdu = observatory.generate_fits_image(
            ra=ra,  # degrees
            dec=dec,  # degrees
            exp_time=exp_time,  # seconds
            dateobs=dateobs,
            sources=sources,
            fwhm_multiplier=20,  # to determine the rendering radius around each star
            user_header={
                "FILTER": filters[i].name,
                "IMAGETYP": "LIGHT",
                "object": cluster_name,
            },
        )

        hdu.writeto(
            f"../data/{cluster_name}/{cluster_name}_{filters[i].name}_Light_{dateobs.strftime('%Y%m%dT%H%M%S.%f')}.fits",
            overwrite=True,
        )

# Calibration frames (bias, dark, flat)
os.makedirs(f"../data/{cluster_name}/calibrations", exist_ok=True)
for j in range(n_imgs):
    dateobs = datetime.now(UTC) + timedelta(seconds=j * (exp_time + 1))

    ## Bias frame
    hdu_bias = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=0,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "BIAS"},
    )
    hdu_bias.writeto(
        f"../data/{cluster_name}/calibrations/Bias_{dateobs.strftime('%Y%m%dT%H%M%S.%f')}.fits",
        overwrite=True,
    )

    ## Dark frame
    hdu_dark = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=exp_time,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "DARK"},
    )
    hdu_dark.writeto(
        f"../data/{cluster_name}/calibrations/Dark_{dateobs.strftime('%Y%m%dT%H%M%S.%f')}.fits",
        overwrite=True,
    )

    for i in range(len(filters)):
        ## Flat frame
        site.sun_altitude = -4
        hdu_flat = observatory.generate_fits_image(
            ra=ra,  # degrees
            dec=dec,  # degrees
            exp_time=0.5,  # seconds
            dateobs=dateobs,
            light=1,
            sources=cabaret.Sources.from_arrays(ra=[], dec=[], fluxes=[]),
            user_header={"FILTER": filters[i].name, "IMAGETYP": "FLAT"},
        )
        hdu_flat.writeto(
            f"../data/{cluster_name}/calibrations/Flat_{filters[i].name}_{dateobs.strftime('%Y%m%dT%H%M%S.%f')}.fits",
            overwrite=True,
        )
        site.sun_altitude = None